# 08 · Full fine-tuning, and what it costs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/08_finetune_full.ipynb)

Companion notebook for *Chronos-2 in Practice (4/5)*. **Needs a GPU**, and
[`07_finetune_lora`](07_finetune_lora.ipynb) must have run first: this notebook reuses its
adapters and its results.

**What you will learn**

- `finetune_mode="full"`: every weight trained, at a learning rate two orders of magnitude lower
  than LoRA's, and why that is not optional.
- The comparison table the decision actually needs: error, training time, peak VRAM, artifact
  size and inference latency, side by side.
- Catastrophic forgetting, measured rather than assumed: all three models evaluated on a domain
  the fine-tune never saw.

Runtime: ~25 min on an RTX 3070 (FAST_MODE: ~5 min).

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time

import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import spain_energy_frame, spain_weather_frame, validate_and_prepare
from c2p.eval import backtest, evaluate, skill_table, summarize
from c2p.experiments import FREQ, LOAD, TRAIN_END, day_ahead_config, weekly_sample_config
from c2p.finetune import (
    finetune,
    load_finetuned,
    temporal_split,
    to_fit_inputs,
    to_univariate_series,
    window_series,
)
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plots.evaluation import plot_skill_scores
from c2p.plotting import save_figure, set_style
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

if DEVICE != "cuda":
    print("WARNING: no GPU visible; full fine-tuning on a CPU is not practical. Use Colab.")
for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")

## 1. The same split, the same data

Everything about the data is identical to notebook `07`, which is what makes the two runs
comparable: same 26 univariate Spanish series, same train/validation/test periods, same horizon
and context.

In [ ]:
HORIZON = 24
CONTEXT = 2048
STEPS = 50 if FAST_MODE else 1000
BATCH = 32

raw = spain_energy_frame()
numeric = [column for column in raw.columns if column not in (ID_COL, TIMESTAMP_COL)]
frame, report = validate_and_prepare(raw[[ID_COL, TIMESTAMP_COL, *numeric]], freq=FREQ, target=LOAD)
univariate = to_univariate_series(frame, numeric)
split = temporal_split(
    univariate, val_start=TRAIN_END + pd.Timedelta(hours=1), test_start="2018-01-01"
)
train_inputs = to_fit_inputs(split.train, freq=FREQ, prediction_length=HORIZON)
val_frame = window_series(
    split.train, split.val, prediction_length=HORIZON, freq=FREQ, step=168, max_context=CONTEXT
)
val_inputs = to_fit_inputs(val_frame, freq=FREQ, prediction_length=HORIZON)

TEST_FRAME = frame[[ID_COL, TIMESTAMP_COL, LOAD]]
TEST_CONFIG = day_ahead_config(FAST_MODE, max_context=CONTEXT)
print(
    f"{len(train_inputs)} training series, {len(val_inputs)} validation windows, "
    f"{TEST_CONFIG.n_windows} test windows"
)

## 2. Training every weight

Two changes from LoRA, and only one of them is a hyperparameter:

- **`mode="full"`**: all 119.5M parameters get gradients, instead of about 1%.
- **A learning rate of 1e-6**, a hundredth of LoRA's. This is AutoGluon's own suggestion for
  full fine-tuning and it is not a knob to shrug at: with the pretrained weights themselves
  moving, a rate that works for a small adapter will walk the model off the representation it
  spent its pretraining acquiring.

The cost shows up in memory rather than in time (optimizer state for every parameter), which is why peak VRAM is the column to watch in the table below.

In [ ]:
full = finetune(
    load_chronos2(device=DEVICE),
    train_inputs,
    val_inputs=val_inputs,
    mode="full",
    prediction_length=HORIZON,
    num_steps=STEPS,
    learning_rate=1e-6,
    batch_size=BATCH,
    context_length=CONTEXT,
    output_dir=PATHS.models / "full_lr1e-6",
    early_stopping_patience=3,
    eval_steps=10 if FAST_MODE else 100,
    overwrite=True,
)
print(
    f"full: {full.trainable_parameters:,} trainable "
    f"({full.trainable_parameters / full.total_parameters:.0%} of the model), "
    f"{full.train_seconds / 60:.1f} min, peak VRAM {full.peak_vram_mb:.0f} MB, "
    f"checkpoint {full.artifact_mb:.0f} MB, best step {full.best_step}"
)
full_costs = pd.DataFrame(
    [
        {
            "run": "full",
            "rank": pd.NA,
            "trainable_parameters": full.trainable_parameters,
            "total_parameters": full.total_parameters,
            "train_seconds": full.train_seconds,
            "peak_vram_mb": full.peak_vram_mb,
            "artifact_mb": full.artifact_mb,
            "steps": full.steps,
            "best_step": full.best_step,
            "best_eval_loss": full.best_eval_loss,
        }
    ]
)
print("Saved", save_results(full_costs, "08_full_costs", PATHS).name)

## 3. Three models, one table

The LoRA adapter with the best test WQL from notebook `07` is reloaded from disk, so all three
models are scored in the same session on the same windows. `seconds` is the wall-clock time of
one window's `predict` call, which is the latency an online service would see.

In [ ]:
lora_skill = load_results("07_lora_skill", PATHS)
best_lora = lora_skill[lora_skill["model"] != "zero-shot"].iloc[0]["model"]
best_rank = int(str(best_lora).removeprefix("lora_r"))
print(f"Best LoRA run from notebook 07: {best_lora}")

MODELS = {
    "zero-shot": Chronos2Forecaster(load_chronos2(device=DEVICE), name="zero-shot"),
    best_lora: Chronos2Forecaster(
        load_finetuned(PATHS.models / f"lora_r{best_rank}" / "finetuned-ckpt", device=DEVICE),
        name=str(best_lora),
    ),
    "full": Chronos2Forecaster(full.pipeline, name="full"),
}

start = time.perf_counter()
frames = []
for name, model in MODELS.items():
    metrics = evaluate(backtest(model, TEST_FRAME, TEST_CONFIG), TEST_FRAME, TEST_CONFIG)
    frames.append(metrics)
    print(
        f"  {name:<12} WQL {metrics['WQL'].mean():.4f}  "
        f"latency {metrics['seconds'].median() * 1000:.0f} ms/window"
    )
compare = pd.concat(frames, ignore_index=True)
print(f"Evaluated in {(time.perf_counter() - start) / 60:.1f} min")
print("Saved", save_results(compare, "08_comparison_metrics", PATHS).name)

In [ ]:
compare = load_results("08_comparison_metrics", PATHS)
compare_skill = skill_table(compare, baseline="zero-shot", metric="WQL")
print("Saved", save_results(compare_skill, "08_comparison_skill", PATHS).name)

best = compare_skill[compare_skill["model"] != "zero-shot"].iloc[0]
verb = "beats" if best["skill"] > 0 else "fails to beat"
title = (
    f"On this dataset fine-tuning {verb} zero-shot: best is {best['model']} at "
    f"{best['skill']:+.1%} [{best['ci_low']:+.1%}, {best['ci_high']:+.1%}]"
)
ax = plot_skill_scores(
    compare_skill,
    highlight=str(best["model"]),
    title=title,
    xlabel="WQL improvement over the zero-shot model (%)",
)
(figure_path,) = save_figure(ax.figure, "08_comparison", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 4. What did it forget?

A fine-tune trades generality for fit. The Spanish electricity table is all these models were
shown, so the question is what happened to everything else: the dataset ships a convenient
out-of-domain test, hourly temperature for five Spanish cities, a different unit, a different
scale and a different shape, never part of the training data.

A model that has specialised will be worse here. That is the cost of specialising, and it only
matters if you plan to serve one model for several domains: this is exactly the decision this
measurement informs.

In [ ]:
weather, weather_report = validate_and_prepare(
    spain_weather_frame(),
    freq=FREQ,
    target="temp_c",
    duplicates="mean",
    categorical=["weather_main"],
)
weather = weather[[ID_COL, TIMESTAMP_COL, "temp_c"]]
OOD_CONFIG = weekly_sample_config(FAST_MODE, target="temp_c", max_context=CONTEXT)

frames = []
for name, model in MODELS.items():
    metrics = evaluate(backtest(model, weather, OOD_CONFIG), weather, OOD_CONFIG)
    frames.append(metrics)
    print(f"  {name:<12} WQL {metrics['WQL'].mean():.4f} on temperature")
forgetting = pd.concat(frames, ignore_index=True)
print("Saved", save_results(forgetting, "08_forgetting_metrics", PATHS).name)

forgetting_skill = skill_table(forgetting, baseline="zero-shot", metric="WQL")
print("Saved", save_results(forgetting_skill, "08_forgetting_skill", PATHS).name)
forgetting_skill.round(4)

In [ ]:
forgetting_skill = load_results("08_forgetting_skill", PATHS)
worst = forgetting_skill.iloc[-1]
title = (
    f"Out of domain (temperature): {worst['model']} loses {abs(worst['skill']):.1%} "
    f"against the untouched model"
)
ax = plot_skill_scores(
    forgetting_skill,
    highlight=str(worst["model"]),
    title=title,
    xlabel="WQL change on an unseen domain vs the zero-shot model (%)",
)
(figure_path,) = save_figure(ax.figure, "08_forgetting", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 5. The decision table

In [ ]:
costs = pd.concat(
    [load_results("07_lora_costs", PATHS), load_results("08_full_costs", PATHS)], ignore_index=True
).set_index("run")
in_domain = summarize(load_results("08_comparison_metrics", PATHS)).set_index("model")
out_domain = summarize(load_results("08_forgetting_metrics", PATHS)).set_index("model")
skill = load_results("08_comparison_skill", PATHS).set_index("model")
ood_skill = load_results("08_forgetting_skill", PATHS).set_index("model")

header = (
    f"{'model':<12} {'WQL':>8} {'vs 0-shot':>10} {'temp WQL':>9} {'vs 0-shot':>10} "
    f"{'train':>7} {'VRAM':>7} {'artifact':>9} {'latency':>8}"
)
print(header)
for model in in_domain.index:
    cost = costs.loc[model] if model in costs.index else None
    print(
        f"{model:<12} {in_domain.loc[model, 'WQL']:8.4f} {skill.loc[model, 'skill']:+9.1%} "
        f"{out_domain.loc[model, 'WQL']:9.4f} {ood_skill.loc[model, 'skill']:+9.1%} "
        + (
            f"{cost['train_seconds'] / 60:6.1f}m {cost['peak_vram_mb']:6.0f}M "
            f"{cost['artifact_mb']:8.1f}M "
            if cost is not None
            else f"{'--':>7} {'--':>7} {'--':>9} "
        )
        + f"{in_domain.loc[model, 'seconds'] * 1000:7.0f}ms"
    )

**Next:** article 5 leaves the notebooks behind and runs from scripts:
[`bench/bench_cpu_gpu.py`](../bench/bench_cpu_gpu.py) for latency and throughput on CPU and
GPU, [`deploy/`](../deploy/) for three ways to serve the model (and the command that deletes
each one), [`scripts/cost_table.py`](../scripts/cost_table.py) for the cost per million
forecasts, and [`monitoring/build_panel.py`](../monitoring/build_panel.py) for the panel that
tells you when this adapter has stopped working.
